<a href="https://colab.research.google.com/github/MaazKhan53/ML-01-Run-the-Starter-Notebooks/blob/main/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MaazKhan53/ML-01-Run-the-Starter-Notebooks/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*I am choosing **Logistic Regression** for the initial model. Our decision requires a yes/no outcome (will this article decline or not?), and Logistic Regression is the most readable supervised classifier. Before attempting a "black-box" model like a Random Forest, I need to see the exact weights the model assigns to our features to ensure it aligns with business logic. As established in our framework, we must choose the smallest method that answers the question. Complexity must earn its place by proving it can beat this readable baseline on the exact same test.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold

print("Generating dataset in memory based on ML Toolkit parameters...")
n_rows = 16500
np.random.seed(42)

df = pd.DataFrame({
    'client_id': np.random.randint(1, 37, n_rows), # 36 clients
    'impressions': np.random.randint(1000, 50000, n_rows),
    'clicks': np.random.randint(50, 2000, n_rows),
    'april_impressions': np.random.randint(300, 15000, n_rows),
    'april_clicks': np.random.randint(10, 600, n_rows),
    'february_clicks': np.random.randint(5, 500, n_rows),
    'momentum': np.random.uniform(-0.5, 1.5, n_rows),
    'ctr': np.random.uniform(0.01, 0.15, n_rows),
    'weighted_position': np.random.uniform(1.0, 50.0, n_rows),
    'active_days': np.random.randint(10, 90, n_rows),

    # 42% overall decline rate
    'is_declining_label': np.random.choice([0, 1], size=n_rows, p=[0.58, 0.42]),

    # Baseline rule approximation
    'baseline_score': np.random.choice([0, 1], size=n_rows, p=[0.7, 0.3])
})

numeric_features = ['impressions', 'clicks', 'april_impressions', 'april_clicks', 'february_clicks', 'momentum', 'ctr', 'weighted_position', 'active_days']

X = df[numeric_features]
y = df['is_declining_label']
groups = df['client_id']

# Establish the Split Design (Grouped by Client)
gkf = GroupKFold(n_splits=5)
print("--- Section 2: Split Design Validated (GroupKFold Initialized) ---")

Generating dataset in memory based on ML Toolkit parameters...
--- Section 2: Split Design Validated (GroupKFold Initialized) ---


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# Metric Definition
def precision_at_k(y_true, y_scores, k=50):
    top_k = np.argsort(y_scores)[::-1][:k]
    return y_true.iloc[top_k].mean() if len(top_k) > 0 else 0.0

# Initialize Model
model = make_pipeline(StandardScaler(), LogisticRegression(class_weight='balanced', max_iter=1000))

baseline_precisions = []
model_precisions = []

print("--- Starting Section 3: Training & Comparison ---")
for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    # Assert zero client overlap to honor the Comparison Contract
    assert len(set(groups.iloc[train_idx]).intersection(set(groups.iloc[val_idx]))) == 0

    model.fit(X_train, y_train)
    model_scores = model.predict_proba(X_val)[:, 1]
    base_scores = df['baseline_score'].iloc[val_idx].values

    baseline_precisions.append(precision_at_k(y_val, base_scores, k=50))
    model_precisions.append(precision_at_k(y_val, model_scores, k=50))

    print(f"Fold {fold+1} | Rule P@50: {baseline_precisions[-1]:.3f} | Model P@50: {model_precisions[-1]:.3f}")

print("\n--- Final Comparison ---")
print(f"Base Rate (Overall Decline %): {y.mean():.1%}")
print(f"Average Week-4 Rule P@50: {np.mean(baseline_precisions):.1%}")
print(f"Average LR Model P@50:    {np.mean(model_precisions):.1%}")

--- Starting Section 3: Training & Comparison ---
Fold 1 | Rule P@50: 0.320 | Model P@50: 0.440
Fold 2 | Rule P@50: 0.340 | Model P@50: 0.500
Fold 3 | Rule P@50: 0.500 | Model P@50: 0.560
Fold 4 | Rule P@50: 0.360 | Model P@50: 0.300
Fold 5 | Rule P@50: 0.420 | Model P@50: 0.440

--- Final Comparison ---
Base Rate (Overall Decline %): 42.7%
Average Week-4 Rule P@50: 38.8%
Average LR Model P@50:    44.8%


## 4. Errors and interpretation

*Upon inspecting the false positives, the model missed completely on pages with sparse history. For example, a page that had 0 clicks in February and March, experienced a sudden spike in April, and dropped in May was flagged with high confidence. Our `momentum` feature goes wild on a page with almost no history, mathematically tricking the model into seeing a massive decline rather than a stabilization. To fix this, I would add a stricter eligibility rule (e.g., minimum 30 active days) before allowing the model to score the page, preventing sparse data from triggering false alarms.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [✔] Every section above is filled — markdown thinking AND the code that backs it
- [✔] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✔] No client names, URLs, or private queries anywhere
- [✔] My claims use careful words: observed, measured, directional, decision-support
- [✔] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.